# Week 2 Day 2 Lab - Agentic RAG, MCP and Memory (free version)

## Giving the agent a long-term memory

Another 25 claims have arrived at UnderCover, and most of them turn on something you won't find in the claim: when a storm was named, what a device is, or how deep a dive site goes.

We give an agent a long-term memory: Qdrant, a vector database, running as an MCP server. We teach it the facts it's missing, then run the UnderCover agent with and without that memory as experiments in Langfuse.

## 4 parts:

PART 1: The even harder claims

PART 2: A memory over MCP

PART 3: Teaching the agent

PART 4: Experiments in Langfuse

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Before you start</h2>
            <span style="color:#EF476F;">This is the free version of the lab. It uses <code>openrouter/free</code> throughout, so it needs no credit on OpenRouter. The full version with GPT-6 Luna is <a href="week2day2.ipynb">week2day2.ipynb</a>.<br/><br/>
            OpenRouter's free models allow 50 requests a day on an account without credit. With <code>TINY = True</code>, the default here, the experiments run on 5 of the claims and this lab makes about 40 requests. <code>openrouter/free</code> picks a different free model for each request, so expect your results to vary more than mine.<br/><br/>
            You'll need your Langfuse keys from Day 1.<br/><br/>
            Please execute each cell and compare your results with mine. Some cells have code that isn't about AI, like the charts. It's not important to understand those in detail.
            </span>
        </td>
    </tr>
</table>

In [1]:
import json
import os
import shutil
from pathlib import Path
from typing import Literal

import httpx
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.agents.middleware import ToolRetryMiddleware
from langchain_core.tools import tool
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_openai import ChatOpenAI
from langfuse import Evaluation, get_client
from langfuse.langchain import CallbackHandler
from week2day1_payout import Benefit, Line, Payout, work_out_payout

## Connect to OpenRouter and Langfuse

The keys come from the `.env` file in the `undercover` folder, including the Langfuse keys you added on Day 1.

In [2]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
langfuse = get_client()

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

if langfuse.auth_check():
    print("Connected to Langfuse")
else:
    print("No Langfuse keys found. Add them to the .env file")

OpenRouter API key found
Connected to Langfuse


## Part 1: The even harder claims

The new claims are in `data/evals/even_harder`, with their policy records and the handlers' decisions.

In [3]:
EVEN_HARDER = Path("../data/evals/even_harder")


def load(name):
    """Read one of the JSON files of the even harder claims."""
    return json.loads((EVEN_HARDER / name).read_text(encoding="utf-8"))


claims = load("claims.json")
labels = {decision["claim_id"]: decision for decision in load("decisions.json")}
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = Path("../data/policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")

25 claims, 25 labels, 25 policies


In [4]:
rows = []
for claim in claims:
    label = labels[claim["claim_id"]]
    rows.append({
        "claim_id": claim["claim_id"],
        "summary": claim["summary"],
        "amount_claimed": claim["amount_claimed"],
        "decision": label["decision"],
        "payout": label["payout"],
    })

data = pd.DataFrame(rows)
data

,claim_id,summary,amount_claimed,decision,payout
0,CLM-1301,Bahamas trip canceled when Hurricane Marisol s...,1835.00,decline,0.00
1,CLM-1302,Cancun vacation canceled when Hurricane Tobias...,3197.00,approve,2473.00
2,CLM-1303,Stuck in Lisbon for a day by an air traffic co...,282.30,decline,0.00
3,CLM-1304,Flight to Denver canceled by the Meridian cabi...,734.40,approve,300.00
4,CLM-1305,"Ferry to Dublin canceled by Storm Ysolde, two ...",432.30,decline,0.00
5,CLM-1306,Train to Rome canceled by the national rail st...,605.60,approve,427.60
6,CLM-1307,Checked bag lost on the way to Tokyo,1611.95,approve,562.95
7,CLM-1308,Robbed outside a metro station in Barcelona,1329.00,approve,1239.00
8,CLM-1309,Backpack stolen on the Paris metro,771.99,approve,671.99
9,CLM-1310,Rented apartment in Split broken into,833.00,approve,434.00


### A lost bag in Tokyo

Here's a claim on a policy with Electronics Plus, which raises the limit for phones, tablets, laptops, cameras, e-readers and smartwatches. What is a Halden Arc?

In [5]:
tokyo = [claim for claim in claims if claim["claim_id"] == "CLM-1307"][0]

print(tokyo["story"])
pd.DataFrame(tokyo["items"])

My checked bag never arrived in Tokyo. Skylark searched for a month, declared it lost and paid me $400. Everything in it is listed, with receipts for anything over $100.


,description,category,amount
0,Suitcase,luggage,189.00
1,Halden Arc,electronics,899.00
2,Lightweight rain jacket,clothing,165.00
3,Two merino T-shirts,clothing,118.00
4,"Hiking shoes, one pair",clothing,139.95
5,Packing cubes,luggage,54.20
6,Charging cables and adapters,electronics,46.80


This is how the handler decided it.

In [6]:
decision = labels["CLM-1307"]

print(decision["reason"])
pd.DataFrame(decision["calculation"])

The carrier declared the bag lost. The Halden Arc is a pocket projector, which is not on the Electronics Plus list, so it counts at the $300 per-item limit on Premium, and the other lines are within the limit. The items come to $1,012.95, less the carrier's $400 and the $50 deductible, leaving $562.95, inside the $1,500 maximum.


,step,amount
0,Suitcase,189.00
1,"Halden Arc pocket projector, limited to $300.00",300.00
2,Lightweight rain jacket,165.00
3,Two merino T-shirts,118.00
4,Hiking shoes,139.95
5,Packing cubes,54.20
6,Charging cables and adapters,46.80
7,Less what others have paid or refunded,-400.00
8,Less the deductible,-50.00


### The knowledge base

The facts the handlers used are in `data/knowledge.json`: news about storms and strikes, a catalogue of devices, and notes on dive sites, rivers and activity operators. Some of them aren't needed by any claim.

In [7]:
knowledge = json.loads(Path("../data/knowledge.json").read_text(encoding="utf-8"))

pd.set_option("display.max_colwidth", None)
pd.DataFrame(knowledge)

,topic,fact
0,events,"National Hurricane Center, July 9, 2026: Tropical Storm Marisol has formed east of the Leeward Islands. It is forecast to become a hurricane on July 11 and to pass over the Bahamas on July 12 and 13."
1,events,"National Hurricane Center, August 18, 2026: Tropical Storm Tobias has formed in the western Caribbean. It is expected to become a hurricane and to reach the coast of the Yucatan Peninsula near Cancun on August 21."
2,events,"Lisbon, June 20, 2026: Portugal's air traffic controllers have announced a 48 hour strike starting at midnight on July 14. Most flights to and from Lisbon and Porto are expected to be canceled on July 14 and 15."
3,events,"Meridian Air press release, August 5, 2026: The union representing our cabin crew has called a strike for August 12 and 13. We expect to cancel about half of our flights on those two days and will rebook affected passengers."
4,events,"North Atlantic weather bulletin, September 2, 2026: Storm Ysolde has been named. Gales are expected across Ireland, Wales and western Scotland from the morning of September 4, and ferry crossings on the Irish Sea are likely to be canceled."
5,events,"Rome, July 3, 2026: Italy's rail unions have called a 24 hour national rail strike from 9 pm on July 21 to 9 pm on July 22. Most high-speed and regional trains will not run."
6,events,"National Hurricane Center, June 30, 2026: Tropical Storm Nadia has formed in the southern Gulf of Mexico and is expected to come ashore in Tamaulipas, Mexico, on July 2."
7,events,"Northwind Airlines press release, July 15, 2026: The pilots' strike planned for July 24 has been called off after an agreement with the union. All flights will operate as scheduled."
8,events,"Japan Meteorological Agency, August 24, 2026: Typhoon Kaori is expected to pass close to Okinawa on August 27 and 28. Flights and ferries in the Ryukyu Islands are likely to be suspended."
9,events,"Athens, August 19, 2026: Greek ferry workers have called a 24 hour strike for August 27. Ferries between Piraeus and the islands will stay in port."


## Part 2: A memory over MCP

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/info.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">MCP</h2>
            <span style="color:#118AB2;">MCP, the Model Context Protocol, is an open standard for connecting agents to tools. An MCP server offers tools, and an MCP client makes them available to an agent.<br/><br/>
            Qdrant's MCP server has two tools: <code>qdrant-store</code> saves a piece of text in a vector database, and <code>qdrant-find</code> searches it by meaning. We run it on our own machine with <code>uvx</code>, and the database lives in a local folder.
            </span>
        </td>
    </tr>
</table>

### One adjustment for Windows

When Python launches another program, it hands that program a place to write its error messages. Inside a Jupyter notebook on Windows, that place is not a real file, and the library that launches MCP servers trips over it. The cell below points the MCP server's error log somewhere safe before we connect. On Mac and Linux it does nothing at all, so run it either way and carry on.

In [8]:
import sys

if sys.platform == "win32":
    import subprocess
    from functools import partial
    import langchain_mcp_adapters.sessions as mcp_sessions

    mcp_sessions.stdio_client = partial(mcp_sessions.stdio_client, errlog=subprocess.DEVNULL)
    print("Applied the Windows adjustment")
else:
    print("Not Windows, so nothing to do here")

Applied the Windows adjustment


### Connecting to the memory server

Qdrant keeps its database in the folder `qdrant_data`. We delete it first, so the agent starts with an empty memory. The first time you connect, `uvx` downloads the server and a small embedding model, which takes about half a minute.

In [9]:
MEMORY = Path("qdrant_data").resolve()
shutil.rmtree(MEMORY, ignore_errors=True)

In [10]:
MEMORY_ENV = {"QDRANT_LOCAL_PATH": str(MEMORY), "COLLECTION_NAME": "memory"}

parameters = {"transport": "stdio", "command": "uvx", "args": ["mcp-server-qdrant@0.8.1"], "env": MEMORY_ENV}

client = MultiServerMCPClient({ "memory": parameters})

memory_tools = await client.get_tools()
for memory_tool in memory_tools:
    print(f"{memory_tool.name}: {memory_tool.description}\n")

qdrant-find: Look up memories in Qdrant. Use this tool when you need to: 
 - Find memories by their content 
 - Access memories for further analysis 
 - Get some personal information about the user

qdrant-store: Keep the memory for later use, when you are asked to remember something.



Each tool call starts the server, runs the tool and stops the server again. Qdrant's local mode lets only one process open the database at a time, so when the agent searches twice at once, one of the searches fails. `ToolRetryMiddleware` tries a failed tool call again after a short wait.

### An agent with a memory

The memory tools come from the server; the instructions tell the agent how to use them.

The MCP tools only run asynchronously so we need to use async python. In Part 4, Langfuse runs each experiment on its own event loop, and a connection opened on one event loop can't be reused on another, so we give the model an HTTP client that doesn't keep connections open between calls.

In [11]:
MEMORY_INSTRUCTIONS = """You have a long-term memory.
When you are told a fact, store it in your memory word for word.
When you are asked a question, search your memory and answer from what you find. If your memory has nothing on it, say that you don't know.
"""

http_client = httpx.AsyncClient(limits=httpx.Limits(max_keepalive_connections=0))
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5, http_async_client=http_client)

assistant = create_agent(model=llm, tools=memory_tools, system_prompt=MEMORY_INSTRUCTIONS, middleware=[ToolRetryMiddleware(max_retries=3)])

`ask` starts a new conversation every time, so the agent can't answer from earlier messages. Whatever it knows next time has to come from its memory.

In [12]:
async def ask(message):
    """Send a message to the assistant in a new conversation and return its reply."""
    result = await assistant.ainvoke({"messages": [{"role": "user", "content": message}]})
    return result["messages"][-1].text


print(await ask("What kind of device is the Halden Arc?"))
print(await ask("When was Tropical Storm Marisol named?"))



I don't have information about the Halden Arc in my memory.


I don't have any information about when Tropical Storm Marisol was named in my memory.


## Part 3: Teaching the agent

In the full version of this lab, we tell the agent each fact in the knowledge base, one message at a time, and it stores each one with `qdrant-store`. That takes more requests than the free daily limit, so here we call `qdrant-store` ourselves for each fact.

In [13]:
store = [memory_tool for memory_tool in memory_tools if memory_tool.name == "qdrant-store"][0]

for entry in knowledge:
    await store.ainvoke({"information": entry["fact"]})
    print(f"Stored: {entry['fact'][:80]}...")

Stored: National Hurricane Center, July 9, 2026: Tropical Storm Marisol has formed east ...


Stored: National Hurricane Center, August 18, 2026: Tropical Storm Tobias has formed in ...


Stored: Lisbon, June 20, 2026: Portugal's air traffic controllers have announced a 48 ho...


Stored: Meridian Air press release, August 5, 2026: The union representing our cabin cre...


Stored: North Atlantic weather bulletin, September 2, 2026: Storm Ysolde has been named....


...

Stored: Talon Excursions, Queenstown, New Zealand: jet-boat rides through the canyons of...


Stored: Blue Chimney, Cayo Brava, Honduras: a cavern dive that runs from 6 meters down t...


Stored: Upper Rio Cuyo, Costa Rica: grade 5 rapids, run only by expert kayakers....


Stored: Glacier Air, Interlaken, Switzerland: helicopter sightseeing flights over the Ju...


Stored: Highline Canyoning, Interlaken, Switzerland: guided canyoning trips with abseils...


We ask the same questions again, each in a new conversation.

In [14]:
print(await ask("What kind of device is the Halden Arc?"))
print(await ask("When was Tropical Storm Marisol named?"))

Based on what I have in my memory, the **Halden Arc** is a **pocket projector** — a small device that can throw a picture of up to 100 inches from a phone or laptop. It had a list price of $899.


A tool is still a function, so we can search the memory ourselves. The search works on meaning, so a question can find a fact that shares none of its words. It returns the 10 closest facts, nearest first.

In [15]:
search = [memory_tool for memory_tool in memory_tools if memory_tool.name == "qdrant-find"][0]

results = await search.ainvoke({"query": "a gadget for watching films on a wall"})
for line in json.loads(results[0]["text"]):
    print(line)

Results for the query 'a gadget for watching films on a wall'
<entry><content>Halden Arc: a pocket projector that throws a picture of up to 100 inches from a phone or laptop. List price $899.</content><metadata></metadata></entry>
<entry><content>Vantor Halo: a titanium smart ring that tracks sleep, heart rate and activity. List price $449.</content><metadata></metadata></entry>
<entry><content>Lumora Drift 3: an e-reader with a 7 inch color E Ink screen and a battery that lasts six weeks. List price $329.</content><metadata></metadata></entry>
<entry><content>Corvane Lumen X1: the earlier Corvane smartwatch, released in 2024 at a list price of $449.</content><metadata></metadata></entry>
<entry><content>Lumora Drift 2: the earlier Lumora e-reader, with a 6 inch black and white screen. List price $199.</content><metadata></metadata></entry>
<entry><content>Halden Beam: a portable Bluetooth speaker with a built-in lamp. List price $199.</content><metadata></metadata></entry>
<entry><con

## Part 4: Experiments in Langfuse

### The UnderCover agent

The instructions, the prompt, `record_decision` and the payout tool are the same as on Day 1.

In [16]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.

Use your tools to record your decision.
"""

PAYOUT_TOOL_LINE = "Use the payout tool to work out the payout on an approved claim, and record the payout it returns.\n"


def make_user_prompt(claim):
    """The customer's policy record and the claim, as JSON."""
    policy = policies[claim["policy_id"]]
    return f"""# Policy record

{json.dumps(policy, indent=2)}

# Claim

{json.dumps(claim, indent=2)}
"""

In [17]:
recorded = {}


@tool(parse_docstring=True)
def record_decision(claim_id: str, rationale: str, decision: Literal["approve", "decline", "refer"], payout: float) -> str:
    """Record the decision on a claim in the claims system.

    Args:
        claim_id: The claim's id
        rationale: 2-3 sentences to explain the decision along with the relevant section of the policy
        decision: The decision on the claim
        payout: The amount to pay in US dollars, or 0 for a decline or a referral
    """
    if claim_id in recorded:
        return f"A decision is already recorded on {claim_id}, and decisions are final"
    if decision == "approve" and payout <= 0:
        return "An approval needs a payout above zero"
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"


@tool(parse_docstring=True)
def calculate_payout(policy_id: str, benefit: Benefit, lines: list[Line], paid_by_others: float = 0.0) -> Payout:
    """Work out the payout on an approved claim with UnderCover's payout rules.

    Args:
        policy_id: The customer's policy id
        benefit: The benefit the claim is paid under
        lines: The covered items and expenses, one for each line of the claim, with any excluded ones left out. A pair or a set counts as one item
        paid_by_others: Anything a carrier, hotel or other party has paid or refunded for the same loss

    Returns:
        The calculation, the payout and whether it needs sign-off
    """
    return work_out_payout(policies[policy_id], benefit, lines, paid_by_others)

### Two agents

Both agents have the payout tool. The second one also gets `qdrant-find`, so it can search the memory whenever it decides to. This is agentic RAG: the agent chooses when to retrieve and what to search for. Its server runs in read-only mode, so it can search the memory but can't change it.

In [18]:
read_only_client = MultiServerMCPClient({
    "memory": {"transport": "stdio", "command": "uvx", "args": ["mcp-server-qdrant@0.8.1"], "env": {**MEMORY_ENV, "QDRANT_READ_ONLY": "true"}}
})

memory_search = await read_only_client.get_tools()
print([memory_tool.name for memory_tool in memory_search])

['qdrant-find']


In [19]:
MEMORY_LINE = "Before you decide, search your memory for any product, event or place in the claim that you need to know more about.\n"

payout_agent = create_agent(model=llm, tools=[record_decision, calculate_payout], system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}\n{policy_wording}\n")

memory_agent = create_agent(
    model=llm,
    tools=[record_decision, calculate_payout, *memory_search],
    system_prompt=f"{INSTRUCTIONS}{PAYOUT_TOOL_LINE}{MEMORY_LINE}\n{policy_wording}\n",
    middleware=[ToolRetryMiddleware(max_retries=3)],
)

### Scoring

The scores are the ones from Day 1, with one more for retrieval. For the claims that need a fact from the knowledge base, `found_fact` is whether the agent's searches brought that fact back.

In [20]:
DECISIONS = ["approve", "decline", "refer"]
WRONGFUL_DECLINE = 1500
NEEDLESS_REFERRAL = 50


def is_right(answer, expected):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == expected["decision"] and abs(answer["payout"] - expected["payout"]) <= 1


def mistake_cost(answer, expected):
    """What the agent's answer costs UnderCover, against the handler's decision."""
    if answer["decision"] == expected["decision"]:
        return abs(answer["payout"] - expected["payout"]) if answer["decision"] == "approve" else 0
    if answer["decision"] == "decline":
        return WRONGFUL_DECLINE
    if answer["decision"] == "approve":
        return answer["payout"]
    return NEEDLESS_REFERRAL


def found_fact(answer, expected):
    """For a claim that needs a fact from the knowledge base, whether the agent's searches found it."""
    if not expected["needs"]:
        return None
    return all(fact in answer["retrieved"] for fact in expected["needs"])


def kappa(table):
    """Cohen's kappa for the agent's decisions against the handlers'."""
    observed = (table["agent"] == table["handler"]).mean()
    agent_share = table["agent"].value_counts(normalize=True)
    handler_share = table["handler"].value_counts(normalize=True)
    expected = sum(agent_share.get(decision, 0) * handler_share.get(decision, 0) for decision in DECISIONS)
    return (observed - expected) / (1 - expected)


def show_confusion_matrix(table, title):
    """Draw the confusion matrix, with the handler's decision on each row and the agent's in each column."""
    matrix = pd.crosstab(table["handler"], table["agent"]).reindex(index=DECISIONS, columns=DECISIONS, fill_value=0)
    fig = px.imshow(matrix, text_auto=True, color_continuous_scale="Blues", labels={"x": "Agent", "y": "Handler", "color": "Claims"}, title=title)
    fig.show()

In [21]:
def score_claim(*, output, expected_output, **kwargs):
    """Item evaluator: whether the agent got the claim right, the cost of its mistake, and whether it found the fact it needed."""
    evaluations = [
        Evaluation(name="right", value=int(is_right(output, expected_output))),
        Evaluation(name="mistake_cost", value=mistake_cost(output, expected_output)),
    ]
    found = found_fact(output, expected_output)
    if found is not None:
        evaluations.append(Evaluation(name="found_fact", value=int(found)))
    return evaluations


def score_table(item_results):
    """One row for each claim in a run: the agent's answer beside the handler's, with the scores."""
    rows = []
    for result in item_results:
        answer, expected = result.output, result.item.expected_output
        rows.append({
            "claim_id": result.item.id,
            "handler": expected["decision"],
            "agent": answer["decision"],
            "handler_payout": expected["payout"],
            "agent_payout": answer["payout"],
            "right": is_right(answer, expected),
            "mistake_cost": mistake_cost(answer, expected),
            "found_fact": found_fact(answer, expected),
            "rationale": answer["rationale"],
        })
    return pd.DataFrame(rows).sort_values("claim_id")


def run_scores(table):
    """The scores for a whole run."""
    agent_approvals = table[table["agent"] == "approve"]
    handler_approvals = table[table["handler"] == "approve"]
    return {
        "decision_accuracy": (table["agent"] == table["handler"]).mean(),
        "accuracy": table["right"].mean(),
        "precision": (agent_approvals["handler"] == "approve").mean(),
        "recall": (handler_approvals["agent"] == "approve").mean(),
        "kappa": kappa(table),
        "cost_of_mistakes": table["mistake_cost"].sum(),
        "straight_through": (table["agent"] != "refer").mean(),
        "found_fact": table["found_fact"].dropna().astype(float).mean(),
    }


def score_run(*, item_results, **kwargs):
    """Run evaluator: the scores for the whole run."""
    scores = run_scores(score_table(item_results))
    return [Evaluation(name=name, value=float(value)) for name, value in scores.items()]


def average_scores(table):
    """The scores for each run of an experiment, averaged over the runs."""
    return pd.DataFrame([run_scores(run_table) for _, run_table in table.groupby("run")]).mean()

### The dataset

We upload the even harder claims as a Langfuse dataset, with the handler's decision and the facts each claim needs as the expected output.

In [22]:
DATASET = "even-harder-claims"

langfuse.create_dataset(name=DATASET, description="UnderCover's even harder claims with the handlers' decisions")

for claim in claims:
    label = labels[claim["claim_id"]]
    langfuse.create_dataset_item(
        dataset_name=DATASET,
        id=claim["claim_id"],
        input=claim,
        expected_output={"decision": label["decision"], "payout": label["payout"], "reason": label["reason"], "needs": label["needs"]},
        metadata={"tiny": label["tiny"]},
    )

With `TINY = True`, the default here, we run 5 of the 25 claims to stay inside the free daily limit. `N_TIMES` is the number of times each experiment runs, so leave it at 1.

In [23]:
TINY = True
N_TIMES = 1

dataset = langfuse.get_dataset(DATASET)
items = dataset.items
if TINY:
    items = [item for item in items if item.metadata["tiny"]]

print(f"Running {len(items)} of the {len(dataset.items)} claims")

Running 5 of the 25 claims


### The task

`decide` uses `ainvoke`, since the memory tools only run asynchronously, and Langfuse runs async tasks too. Along with the decision, `decide` keeps what the agent's memory searches returned.

In [24]:
langfuse_handler = CallbackHandler()


async def decide(agent, claim):
    """Run an agent on one claim and return its decision and what its memory searches returned."""
    result = await agent.ainvoke(
        {"messages": [{"role": "user", "content": make_user_prompt(claim)}]},
        config={"callbacks": [langfuse_handler]},
    )
    answer = recorded.get(claim["claim_id"], {"decision": "refer", "payout": 0.0, "rationale": "Nothing recorded"})
    retrieved = "\n".join(message.text for message in result["messages"] if message.type == "tool" and message.name == "qdrant-find")
    print(f"{claim['claim_id']}: {answer['decision']} ${answer['payout']:,.2f}")
    return {**answer, "retrieved": retrieved}


async def payout_task(*, item, **kwargs):
    """Run the agent with the payout tool on one claim from the dataset."""
    return await decide(payout_agent, item.input)


async def memory_task(*, item, **kwargs):
    """Run the agent with the payout tool and memory on one claim from the dataset."""
    return await decide(memory_agent, item.input)

### Experiment 1: the payout tool

This is the agent from Day 1 with the payout tool.

In [25]:
payout_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    result = langfuse.run_experiment(
        name="Payout tool",
        description="The agent with the payout tool",
        data=items,
        task=payout_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(result.format())
    payout_tables.append(score_table(result.item_results).assign(run=run))

CLM-1320: approve $663.67


CLM-1317: refer $0.00


CLM-1314: refer $0.00


CLM-1307: approve $1,161.95


CLM-1301: approve $1,223.00


Individual Results: Hidden (5 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: Payout tool
📋 Run name: Payout tool - 2026-10-23T13:21:47.551928Z - The agent with the payout tool
5 items
Evaluations:
  • mistake_cost
  • found_fact
  • right

Average Scores:
  • mistake_cost: 384.400
  • found_fact: 0.000
  • right: 0.200

Run Evaluations:
  • decision_accuracy: 0.400
  • accuracy: 0.200
  • precision: 0.667
  • recall: 0.667
  • kappa: 0.063
  • cost_of_mistakes: 1922.000
  • straight_through: 0.600
  • found_fact: 0.000

🔗 Experiment:
   https://cloud.langfuse.com/project/cmvjhnxbi001lad0knjaa2n4h/experiments/results?baseline=9e111250dab745ed


In [26]:
payout_table = pd.concat(payout_tables, ignore_index=True)
show_confusion_matrix(payout_table, "Payout tool")

In [27]:
len(payout_table[~payout_table["right"]])

4

### Experiment 2: the payout tool and memory

The same agent, now able to search its memory.

In [28]:
memory_tables = []
for run in range(1, N_TIMES + 1):
    recorded.clear()
    result = langfuse.run_experiment(
        name="Payout tool and memory",
        description="The agent with the payout tool and a memory of the knowledge base",
        data=items,
        task=memory_task,
        evaluators=[score_claim],
        run_evaluators=[score_run],
        max_concurrency=1,
    )
    print(result.format())
    memory_tables.append(score_table(result.item_results).assign(run=run))

CLM-1320: refer $0.00


CLM-1317: approve $1,173.90


CLM-1314: decline $0.00


CLM-1307: approve $1,161.95


CLM-1301: approve $1,223.00


Individual Results: Hidden (5 items)
💡 Set include_item_results=True to view them

──────────────────────────────────────────────────
🧪 Experiment: Payout tool and memory
📋 Run name: Payout tool and memory - 2026-10-23T13:31:15.294511Z - The agent with the payout tool and a memory of the knowledge base
5 items
Evaluations:
  • mistake_cost
  • found_fact
  • right

Average Scores:
  • mistake_cost: 374.400
  • found_fact: 0.750
  • right: 0.400

Run Evaluations:
  • decision_accuracy: 0.600
  • accuracy: 0.400
  • precision: 0.667
  • recall: 0.667
  • kappa: 0.286
  • cost_of_mistakes: 1872.000
  • straight_through: 0.800
  • found_fact: 0.750

🔗 Experiment:
   https://cloud.langfuse.com/project/cmvjhnxbi001lad0knjaa2n4h/experiments/results?baseline=37d071cf8a2e87c1


In [29]:
memory_table = pd.concat(memory_tables, ignore_index=True)
show_confusion_matrix(memory_table, "Payout tool and memory")

In [30]:
memory_table[~memory_table["right"]]

,claim_id,handler,agent,handler_payout,agent_payout,right,mistake_cost,found_fact,rationale,run
0,CLM-1301,decline,approve,0.00,1223.00,False,1223.0,False,"The claim is for trip cancellation due to Hurricane Marisol, which halted Skylark flights for over 24 hours, meeting the severe weather condition in Section 2 of the Trip Cancellation and Interruption benefit. All required documents (cancellation notice, resort letter, receipts) are provided, and the prepaid costs are not refundable. After subtracting the airline voucher credit of $612, the eligible amount is $1,223, which is below the $5,000 maximum for the premium plan. Therefore the claim is approved with a payout of $1,223.",1
1,CLM-1307,approve,approve,562.95,1161.95,False,599.0,True,"Claim covered under Section 5 (Baggage and Personal Effects): checked bag lost by carrier during covered trip, carrier report and settlement letter provided. Halden Arc is a pocket projector (electronics) covered under the Electronics Plus add-on at $899, under the $1,200 per-item limit. Carrier payment of $400 and $50 Premium deductible deducted. Payout: $1,161.95.",1
4,CLM-1320,approve,refer,663.67,0.00,False,50.0,None,Nothing recorded,1


Open a trace from this experiment in Langfuse to see what the agent searched for and what came back.

### Comparing the experiments

In [31]:
pd.DataFrame([average_scores(payout_table), average_scores(memory_table)], index=["Payout tool", "Payout tool and memory"]).round(3)

,decision_accuracy,accuracy,precision,recall,kappa,cost_of_mistakes,straight_through,found_fact
Payout tool,0.4,0.2,0.667,0.667,0.063,1922.0,0.6,0.00
Payout tool and memory,0.6,0.4,0.667,0.667,0.286,1872.0,0.8,0.75


In Langfuse, open **Datasets** and then `even-harder-claims`, and select the runs to compare them claim by claim.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#118AB2;">Commercial implications</h2>
            <span style="color:#118AB2;">Agentic RAG allows you to give an agent expertise about your business, and to build up its knowledge over time (long-term memory).
            Consider the expertise that you could equip an agent with for your business.
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#EF476F;">Your assignment</h2>
            <span style="color:#EF476F;">The lab uses about 40 of the 50 free requests a day, so come back to these tomorrow.<br/><br/>
            Challenge: replace Qdrant with a simple word lookup. Write a tool that searches the facts in <code>data/knowledge.json</code> for the words in its query, give it to the memory agent in place of <code>qdrant-find</code>, and run Experiment 2 again. <code>decide</code> picks out the search results by the tool's name, so change <code>qdrant-find</code> there to the name of your tool. How do the scores compare with searching by meaning?<br/><br/>
            Then give the agent a memory about each customer: a Markdown file for each one, named by the claim's <code>customer_id</code>, with a tool to read a customer's notes and a tool to add to them. Each customer has only one claim in this set, so write a few notes yourself to start with. If you run an experiment again, what should happen to the notes between runs?<br/><br/>
            Stretch challenge: move the retrieval into a sub-agent that has <code>qdrant-find</code> and the customer memory tools, and let the UnderCover agent call the sub-agent as a tool.<br/><br/>
            I'd love it if you would consider making a PR with your version of the lab in the <code>community_contributions</code> folder. There are instructions on making a PR at <a href="https://edwarddonner.com/pr">edwarddonner.com/pr</a>.
            </span>
        </td>
    </tr>
</table>

## Cleaning up Langfuse

This last step is optional. Set `CLEAN_UP = True` and run the cell to delete every trace in the Langfuse project your keys belong to, along with the experiments built on them, and the claims in the `even-harder-claims` dataset. This can't be undone. Only do it in a project you made for this course.

The empty `even-harder-claims` dataset stays, and running Part 4 again fills it. The run scores (accuracy, kappa and the rest) stay too. Langfuse's free plan allows 50 deletions of trace data a day, and each score would take one.

In [32]:
def all_trace_ids():
    """Every trace id in the Langfuse project, collected from its observations a page at a time."""
    trace_ids = set()
    cursor = None
    while True:
        page = langfuse.api.observations.get_many(fields="core", limit=1000, cursor=cursor)
        trace_ids.update(observation.trace_id for observation in page.data)
        cursor = page.meta.cursor
        if not cursor:
            return list(trace_ids)


CLEAN_UP = False

if CLEAN_UP:
    trace_ids = all_trace_ids()
    for start in range(0, len(trace_ids), 1000):
        langfuse.api.trace.delete_multiple(trace_ids=trace_ids[start:start + 1000])

    dataset_items = langfuse.get_dataset(DATASET).items
    for item in dataset_items:
        langfuse.api.dataset_items.delete(id=item.id)

    print(f"Deleted {len(trace_ids)} traces and {len(dataset_items)} dataset items")